In [6]:
import kagglehub
import pandas as pd
import os

# downloads to a local cache directory, returns the path
path = kagglehub.dataset_download("mashlyn/online-retail-II-uci")
print(path)
print(os.listdir(path))

/Users/ryanmccurry/.cache/kagglehub/datasets/mashlyn/online-retail-II-uci/versions/3
['online_retail_II.csv']


In [47]:
df = pd.read_csv(os.path.join(path, "online_retail_II.csv"))

print(f'Shape: {df.shape}')
df.head()

Shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [15]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [44]:
cancellations = df[df['Invoice'].astype(str).str.startswith('C')]
print(f'Number of cancellation rows: {len(cancellations):,}')
print(f'Pencent of total: {len(cancellations) / len(df) * 100:.2f}%')
cancellations.head()

Number of cancellation rows: 19,494
Pencent of total: 1.83%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [40]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Start date:         {df["InvoiceDate"].min().date()}')
print(f'End date:           {df["InvoiceDate"].max().date()}')
print(f'Date range span:    {(df["InvoiceDate"].max() - df["InvoiceDate"].min()).days} days')

Start date:         2009-12-01
End date:           2011-12-09
Date range span:    738 days


In [34]:
n_customers = df['Customer ID'].nunique()
print(f'Number of unique customers: {n_customers}')

Number of unique customers: 5942


In [37]:
invoices_per_customer = df.groupby('Customer ID')['Invoice'].nunique()
print(f'Average invoices per customer:  {invoices_per_customer.mean():.2f}')
print(f'Median invoices per customer:   {invoices_per_customer.median():.2f}')

Average invoices per customer:  7.55
Median invoices per customer:   4.00


In [43]:
df_clean = df.dropna(subset=['Customer ID']).copy()
print(f'Rows before: {len(df):,}')
print(f'Rows after: {len(df_clean):,}')
print(f'Rows dropped: {len(df) - len(df_clean):,}')

Rows before: 1,067,371
Rows after: 824,364
Rows dropped: 243,007


In [ ]:
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['Customer ID'].head()

0    13085
1    13085
2    13085
3    13085
4    13085
Name: Customer ID, dtype: int64

**Cancellation Handling**: Rather than discarding cancelled orders, we split them into a separate dataframe (`cancellations_clean`) to preserve return behavior as a potential churn signal. Purchases and cancellations will be used together during feature engineering (e.g., a customer's return rate may correlate with future churn)

In [51]:
# split cancellations out from actual purchases
cancellations_clean = df_clean[df_clean['Invoice'].astype(str).str.startswith('C')].copy()
purchases_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')].copy()

print(f'Purchases: {len(purchases_clean):,} rows')
print(f'Cancellations: {len(cancellations_clean):,} rows')
print(f'Total (should match df_clean): {len(purchases_clean) + len(cancellations_clean):,}')

Purchases: 805,620 rows
Cancellations: 18,744 rows
Total (should match df_clean): 824,364


In [52]:
missing_description = purchases_clean['Description'].isnull().sum()
print(f'Missing descriptions in purchases_clean: {missing_description}')

Missing descriptions in purchases_clean: 0


In [65]:
last_date = purchases_clean['InvoiceDate'].max()
reference_date = last_date - pd.Timedelta(days=180)
print(f'Last date in date: {last_date}')
print(f'Reference date: {reference_date}')

Last date in date: 2011-12-09 12:50:00
Reference date: 2011-06-12 12:50:00


In [66]:
# split into pre-reference (features) and post-reference (outcome) windows
pre_reference = purchases_clean[purchases_clean['InvoiceDate'] <= reference_date]
post_reference = purchases_clean[purchases_clean['InvoiceDate'] > reference_date]

# customers active before the reference date
active_customers = pre_reference['Customer ID'].unique()
print(f'Active customers as of reference date: {len(active_customers):,}')

# customers who purchased again in the 90-day window after
returning_customers = post_reference['Customer ID'].unique()
print(f'Customers with purchases within the outcome window: {len(returning_customers):,}')

# build the label
churn_labels = pd.DataFrame({'Customer ID': active_customers})
churn_labels['churned'] = (~churn_labels['Customer ID'].isin(returning_customers)).astype(int)

print(f'\nChurn rate: {churn_labels["churned"].mean():.2%}')
churn_labels['churned'].value_counts()

Active customers as of reference date: 4,979
Customers with purchases within the outcome window: 3,479

Churn rate: 48.24%


churned
0    2577
1    2402
Name: count, dtype: int64

Tested both 90-day and 180-day churn windows. Given the median purchase cycle of ~180 days observed in EDA, a 90-day window over-flags naturally infrequent buyers as churned; a 180-day window better reflects this dataset's actual purchase rhythm.